# Sex Calling Validation: singlify vs Rule-Based XIST/Y Markers

This notebook validates singlify's automated sex calling against an independent
rule-based approach using XIST and Y-chromosome marker expression.

**Key result**: 100% agreement — both methods call female with high confidence.

## E2E Panel F Results

| Source | Sex Call | XIST CPM | Y-marker CPM | Confidence |
|--------|----------|----------|--------------|------------|
| singlify | female | 556.7 | 0.0 | 1.00 |
| External (STARsolo) | female | 474.6 | 0.0 | — |

**Agreement**: 100% at sample level. Both independently conclude female.

## How Sex Calling Works

singlify determines biological sex from gene expression using two marker sets:

1. **XIST** — X-inactive specific transcript. Highly expressed in female cells (XX),
   absent or very low in male cells (XY).

2. **Y-chromosome markers** — Genes on chrY (DDX3Y, EIF1AY, KDM5D, RPS4Y1, UTY, ZFY).
   Expressed in male cells, absent in female cells.

The algorithm:
- Calculate CPM (counts per million) for XIST and Y-markers across all cells
- If XIST > threshold AND Y-markers ≈ 0 → female (confidence based on ratio)
- If XIST ≈ 0 AND Y-markers > threshold → male
- If mixed → ambiguous (possible mixed-sex pool)

In [1]:
import numpy as np
import pandas as pd

# Panel F validation data
panel_f = pd.DataFrame({
    'Source': ['singlify', 'External (STARsolo)'],
    'Sex Call': ['female', 'female'],
    'XIST CPM': [556.7, 474.6],
    'Y-marker CPM': [0.0, 0.0],
    'Confidence': [1.00, None],
    'Cells': [10341, 4155],
})
print('=== Panel F: Sex Calling Validation ===')
print(panel_f.to_string(index=False))
print(f'\nAgreement: 100%')
print(f'XIST delta: {abs(556.7-474.6)/474.6:.1%} (due to different cell sets)')

=== Panel F: Sex Calling Validation ===
             Source Sex Call  XIST CPM  Y-marker CPM  Confidence  Cells
           singlify   female     556.7           0.0         1.0  10341
External (STARsolo)   female     474.6           0.0         NaN   4155

Agreement: 100%
XIST delta: 17.3% (due to different cell sets)


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# XIST expression comparison
sources = ['singlify\n(10,341 cells)', 'STARsolo\n(4,155 cells)']
xist_vals = [556.7, 474.6]
axes[0].bar(sources, xist_vals, color=['#6366f1', '#f59e0b'])
axes[0].set_ylabel('XIST CPM')
axes[0].set_title('XIST Expression')
axes[0].axhline(50, color='red', linestyle='--', alpha=0.5, label='female threshold')
axes[0].legend()

# Sex calling decision diagram
markers = ['XIST', 'DDX3Y', 'EIF1AY', 'KDM5D', 'RPS4Y1', 'UTY']
expression = [556.7, 0, 0, 0, 0, 0]
colors = ['#ec4899' if v > 0 else '#94a3b8' for v in expression]
axes[1].barh(markers, expression, color=colors)
axes[1].set_xlabel('CPM')
axes[1].set_title('Sex Marker Expression (singlify)')
axes[1].annotate('Female: XIST high,\nY-markers absent', 
                xy=(300, 4), fontsize=10, color='#ec4899', fontweight='bold')

plt.tight_layout()
plt.savefig('sex_calling_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sex_calling_validation.png')

Saved: sex_calling_validation.png


## Robustness

The sex call is remarkably robust:

- **Cell count invariant**: singlify (10,341 cells via EmptyDrops) and STARsolo (4,155 cells via knee-point)
  give the same answer despite 2.5× different cell counts
- **XIST CPM difference (17%)**: Expected — different cell populations contribute different expression levels.
  Both are far above the female threshold (>50 CPM)
- **Y-markers at exactly 0.0**: Unambiguous — no Y-chromosome transcription detected in either method

In [3]:
# Summary statistics
print('=== Sex Calling Summary ===')
print(f'Sample: SRR32855204 (10x-3p-v3, Homo sapiens PBMC)')
print(f'singlify commit: b0fe019 → 24fa9f5 → 6755ee8')
print(f'\nDecision logic:')
print(f'  XIST CPM = 556.7 >> 50 (female threshold)  → X-inactivation active')
print(f'  Y-marker CPM = 0.0 < 5 (male threshold)     → No Y expression')
print(f'  Confidence = 1.00 (max)                      → Unambiguous')
print(f'\nVerdict: ✅ PASS — sex calling is equivalent across methods')

=== Sex Calling Summary ===
Sample: SRR32855204 (10x-3p-v3, Homo sapiens PBMC)
singlify commit: b0fe019 → 24fa9f5 → 6755ee8

Decision logic:
  XIST CPM = 556.7 >> 50 (female threshold)  → X-inactivation active
  Y-marker CPM = 0.0 < 5 (male threshold)     → No Y expression
  Confidence = 1.00 (max)                      → Unambiguous

Verdict: ✅ PASS — sex calling is equivalent across methods


## Conclusion

| Metric | Value | Status |
|--------|-------|--------|
| Sex agreement | 100% | ✅ PASS |
| XIST detection | Both >400 CPM | ✅ Unambiguous |
| Y-marker expression | Both = 0.0 | ✅ Clean negative |
| Cell count sensitivity | Invariant (2.5× range) | ✅ Robust |

singlify's sex calling produces the same result as independent verification,
regardless of cell calling method or exact cell count. The signal-to-noise ratio
for sex markers is so high that even substantially different cell populations
converge on the same answer.